In [190]:
import polars as pl
import torch
from sklearn.metrics import classification_report
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Lasso
import numpy as np
from amaretto_mlp import *
from utils_b import *
from autoencoder import *
from explanation import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1. Data & model preperatiom

In [2]:
pl.Config.set_tbl_cols(150) 

polars.config.Config

In [3]:
df = pl.read_parquet("amaretto_transformed_scaled.pq")
df_train, df_val, df_test = split_data(df)

#columns_to_drop = ['id', 'Anomaly', 'Anomaly_bin', 'Originator', 'datetime']
columns_to_drop = ['id', 'Anomaly', 'Anomaly_bin', 'Originator', 'datetime', 'day_of_week', 'day_of_month', 'hour', 'minute']

target_column = 'Anomaly'
y_train = df_train.select(target_column).to_numpy().flatten()
df_train = df_train.drop(columns_to_drop)
X_train = df_train.to_numpy()
y_val = df_val.select(target_column).to_numpy().flatten()
X_val = df_val.drop(columns_to_drop).to_numpy()
y_test = df_test.select(target_column).to_numpy().flatten()
X_test = df_test.drop(columns_to_drop).to_numpy()

train_dataset = AmarettoDataset(X_train, y_train)
val_dataset = AmarettoDataset(X_val, y_val)
test_dataset = AmarettoDataset(X_test, y_test)
feature_names = df_train.columns
print(X_train.shape)
print(X_test.shape)

(24451934, 121)
(2765755, 121)


In [4]:
train_loader = DataLoader(train_dataset, batch_size=256, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=256, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, num_workers=2)
input_dim = X_train.shape[1]
df_train, df_val, df_test = split_data(df)
print(len(train_loader))

95516


In [5]:
model = CombinedModel(input_dim=X_train.shape[1], hidden_layers=[64, 64], latent_dim=16, num_classes=6)
model.load_state_dict(torch.load("model_combined_2.pth"))
print(model)

CombinedModel(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=121, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
      (4): Linear(in_features=64, out_features=16, bias=True)
    )
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
      (4): Linear(in_features=64, out_features=121, bias=True)
    )
  )
  (classifier): Classifier(
    (network): Sequential(
      (0): Linear(in_features=16, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=6, bias=True)
    )
  )
)


In [6]:
model.eval()
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
with torch.no_grad():
    _, y_pred_train, z_train = model(X_train_tensor)
    y_pred_train = torch.argmax(y_pred_train, dim=1)
y_pred_train = y_pred_train.cpu().detach().numpy()
print(y_pred_train.shape)
z_train = z_train.cpu().numpy()
print(z_train.shape)

(24451934,)
(24451934, 16)


In [7]:
model.eval()
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    _, y_pred_test, z_test = model(X_test_tensor)
    y_pred_test = torch.argmax(y_pred_test, dim=1)
y_pred_test = y_pred_test.cpu().detach().numpy()
print(y_pred_test.shape)
z_test = z_test.cpu().numpy()
print(z_test.shape)

(2765755,)
(2765755, 16)


In [8]:
df_test = df_test.with_columns(pl.Series("Model Prediction", y_pred_test))
df_test = df_test.select(["Model Prediction", *df_test.columns[:-1]])  
df_test = df_test.with_row_index(name="index")
df_test

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,0,27025095,2019-03-18 00:00:01,"""Client_023""",0,0,0.507711,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,18,0,0,false,1.771247,1.252211,1.039728,0.94558,-2.511988,1.660603,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-1.175981,-1.290214,-0.000167,0.000056,0.71708,1.365456,0.513469,-1.287407,0.279149,3.620574,0.147873,1.40623,-284.418707,-594.740725,-701.775893,-0.891973,-40.751886,-33.504513,-40.

In [9]:
df_train = df_train.with_columns(pl.Series("Model Prediction", y_pred_train))
df_train = df_train.select(["Model Prediction", *df_train.columns[:-1]])  
df_train = df_train.with_row_index(name="index")
df_train

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,0,151925,2019-01-01 00:00:03,"""Client_259""",0,0,0.392679,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,2,1,0,0,false,-5.176871,-4.005506,-3.292225,-2.835853,-2.511988,-0.669069,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-0.000079,-0.000208,-0.000167,0.000056,-0.811648,0.879612,0.087694,1.476385,0.279149,3.620574,0.147873,1.40623,-284.418707,-594.740725,-701.775893,-1850.737809,-40.751886,-33.5045

In [10]:
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   2759074
           1       0.35      0.84      0.49       576
           2       0.43      0.85      0.57       257
           3       0.70      0.95      0.81      1175
           4       0.41      0.83      0.55      2228
           5       0.52      0.43      0.47      2445

    accuracy                           1.00   2765755
   macro avg       0.57      0.82      0.65   2765755
weighted avg       1.00      1.00      1.00   2765755



In [11]:
mask_correct_preds = (y_test[y_test != 0] == y_pred_test[y_test != 0])
mask_wrong_preds = (y_test[y_test != 0] != y_pred_test[y_test != 0])
print(mask_correct_preds.sum())
print(mask_wrong_preds.sum())

4723
1958


In [12]:
df_correct_preds = df_test.filter((pl.col("Anomaly") != 0))
df_correct_preds = df_correct_preds.filter(mask_correct_preds)
df_wrong_preds = df_test.filter((pl.col("Anomaly") != 0))
df_wrong_preds = df_wrong_preds.filter(mask_wrong_preds)
print(df_correct_preds.height)
print(df_wrong_preds.height)

4723
1958


# 2. Explanations on input space

In [22]:
#X_train_n = X_train[y_pred_train == 0]
y_train_n = y_pred_train[y_pred_train == 0]
print(X_train_n.shape)
print(y_train_n.shape)

(24291851, 121)
(24291851,)


## 2.1. Small but highly frequent transactions generated within a short timeframe:
A pattern that contains multiple transactions below applicable reporting thresholds.  


In [24]:
anomaly_type = 1
idx = 8357	
k = 9

In [25]:
X_train_a = X_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(X_train_a.shape)
print(y_train_a.shape)

(16437, 121)
(16437,)


In [21]:
# z_train_n = z_train[y_train == 0]
# y_train_n = y_pred_train[y_pred_train == 0]
# z_train_a = z_train[y_pred_train == anomaly_type]
# y_train_a = y_pred_train[y_pred_train == anomaly_type]

In [26]:
# x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

# model.encoder.eval()
# test_transaction_latent = model.encoder(x)
# test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
# test_transaction_latent = test_transaction_latent.reshape(1, -1)
# test_transaction_latent.shape

(1, 121)
[[-1.37499299  1.          0.          1.          0.          0.        ]]
1
1


In [39]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8357,1,27181276,2019-03-18 07:07:36,"""Client_126""",1,1,-1.374993,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,1,18,7,7,true,2.040487,1.445817,1.199385,1.077004,1.010305,-0.669069,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,2.410951,-0.183119,-0.071133,0.0,0.0,-1.058838,-0.000208,-0.000167,-2.606994,0.71708,1.365456,0.513469,-1.287407,-1.409974,1.270989,-0.831225,1.143892,-1.830154,-3.55436,-4.216388,-0.910444,-3.529966,-4.415957,-6.225

In [28]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(X_train_n)
distances_n, indices_n = knn_normal.kneighbors(x, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(X_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(x, n_neighbors=k)
print(indices_a)
print(distances_a)

[[14130659 14132990 14132071 14132828 14134052 14133950 14133800  4559815
  14129022]]
[[6.27184505 6.44992011 6.56645748 6.75125626 6.76472677 6.83805775
  6.84874294 6.86661913 6.94610959]]
[[ 5498 10564 10565 10557  8497  8498 14241 14249  9687]]
[[7.39361877 7.57207238 7.71109299 8.01715007 8.21487136 8.51072918
  8.55808045 8.56119751 8.57957322]]


In [25]:
#df_train.filter(pl.col('Anomaly') == 0)[indices_n.flatten().tolist()]

In [26]:
#df_train.filter(pl.col('Anomaly') == 1)[indices_a.flatten().tolist()]

In [29]:
lasso_x_train = np.vstack((X_train_n[indices_n.flatten().tolist()], X_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))
print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 121)
(18,)


In [41]:
lasso_classifier = Lasso(alpha=0.01)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
#print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

Lasso(alpha=0.01)

In [42]:
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[ 13  25   0  71  46  41 120  73  89  32]
Product Type_FX: -0.44062356801067176
Product Class_Cash in / out (withdrawal), Security in / out: 0.43953873509925334
amount_log: -0.17105727996190773
rolling_std_1h: 0.11148772891697573
product_class_index_timestamp_diff_1: -0.10959079743362234
product_type_index_timestamp_diff_1: 0.08762863897469755
Currency_freq_originator: 0.08141598715522942
rolling_std_24h: 0.06801874787144765
rolling_mean_originator_24h: 0.05955602363474946
originator_index_timestamp_diff_2: 0.05845220259692664


In [43]:
imp_features = [feature_names[idx] for idx in sorted_importance]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['Product Type_FX', 'Product Class_Cash in / out (withdrawal), Security in / out', 'amount_log', 'rolling_std_1h', 'product_class_index_timestamp_diff_1', 'product_type_index_timestamp_diff_1', 'Currency_freq_originator', 'rolling_std_24h', 'rolling_mean_originator_24h', 'originator_index_timestamp_diff_2']


Product Type_FX,"Product Class_Cash in / out (withdrawal), Security in / out",amount_log,rolling_std_1h,product_class_index_timestamp_diff_1,product_type_index_timestamp_diff_1,Currency_freq_originator,rolling_std_24h,rolling_mean_originator_24h,originator_index_timestamp_diff_2
u8,u8,f64,f64,f64,f64,f64,f64,f64,f64
0,1,-1.374993,-1.218832,2.410951,-1.390533,0.627225,-2.099412,-6.993079,1.445817


In [36]:
y_pred = lasso_classifier.predict(x.reshape(1, -1))
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[0.53399728]
R squared training set 0.9712493053772452


In [40]:
lasso_classifier = Lasso(alpha=0.05)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)
#print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[  0  13  71  58  41  65  31  46  25 116]
amount_log: -0.22774516507272935
Product Type_FX: -0.20657789590888484
rolling_std_1h: 0.10104766108921424
day_of_month_cos: 0.06589130934568095
product_type_index_timestamp_diff_1: 0.04769999608521683
rolling_mean_24h: 0.045060945086517167
originator_index_timestamp_diff_1: -0.04103741245885048
product_class_index_timestamp_diff_1: -0.02216899866712941
Product Class_Cash in / out (withdrawal), Security in / out: 0.009038137501358446
InputOutput_freq_originator: 0.0038723635540293664


## 2.2. Transactions with rounded normalized amounts bought or sold within an account:
It is unusual for transactions in capital markets to have rounded amounts (unless they occur in markets where foreign exchange conversion causes rounding errors).  


In [198]:
anomaly_type = 2
idx = 2762998
#idx = 8891

In [54]:
X_train_a = X_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(X_train_a.shape)
print(y_train_a.shape)

(3075, 121)
(3075,)


In [55]:
x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

(1, 121)
[[-0.15372607  0.          1.          0.          0.          1.        ]]
2
2


In [56]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2762998,2,29358356,2019-03-25 20:58:44,"""Client_126""",2,1,-0.153726,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,1,25,20,58,true,1.976088,1.405621,1.183206,1.062452,0.996917,1.666501,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-0.92282,-1.812988,-0.000167,0.000056,0.71708,1.365456,1.493561,0.276227,1.227968,1.900558,0.437069,1.344762,-0.20075,-0.763227,-0.89009,-0.796219,-0.016536,1.158688,0.548016

In [57]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(X_train_n)
distances_n, indices_n = knn_normal.kneighbors(x, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(X_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(x, n_neighbors=k)
print(indices_a)
print(distances_a)

[[ 7500287  7487612 19638126  7498911  7479744  7493992  7496824  7504858
  19634012]]
[[5.33582442 5.57497992 5.62633362 5.63051838 5.6475954  5.70801727
  5.72663683 5.75048922 5.77467883]]
[[2847 2852 2274 1049 2816 1043 2056 1434 2679]]
[[5.39437431 6.37619863 6.57638483 6.58323012 6.65733044 6.74744505
  6.79097108 6.87449587 6.93982183]]


In [58]:
lasso_x_train = np.vstack((X_train_n[indices_n.flatten().tolist()], X_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))
print(lasso_x_train.shape)
print(lasso_y_train.shape)
lasso_classifier = Lasso(alpha=0.01)
lasso_classifier.fit(lasso_x_train, lasso_y_train)

(18, 121)
(18,)


Lasso(alpha=0.01)

In [59]:
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[ 30  56  82 116  98  79 114 115 120 112]
is_rounded: 1.8373229624261656
day_of_week_cos: -0.038877705878208546
rolling_min_7d: -0.02488240302674951
InputOutput_freq_originator: 0.014420370847847905
rolling_max_originator_7d: -0.007949514242944685
rolling_min_1h: 0.000842879627505212
cumcount_Product Class: 0.0
cumcount_Currency: 0.0
Currency_freq_originator: 0.0
cumcount_Market: 0.0


In [60]:
imp_features = [feature_names[idx] for idx in sorted_importance]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['is_rounded', 'day_of_week_cos', 'rolling_min_7d', 'InputOutput_freq_originator', 'rolling_max_originator_7d', 'rolling_min_1h', 'cumcount_Product Class', 'cumcount_Currency', 'Currency_freq_originator', 'cumcount_Market']


is_rounded,day_of_week_cos,rolling_min_7d,InputOutput_freq_originator,rolling_max_originator_7d,rolling_min_1h,cumcount_Product Class,cumcount_Currency,Currency_freq_originator,cumcount_Market
bool,f64,f64,f64,f64,f64,f64,f64,f64,f64
true,1.365456,1.387032,2.639691,-0.538934,0.521348,-0.138752,0.069083,0.632801,-0.798103


In [61]:
y_pred = lasso_classifier.predict(x)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[1.87611121]
R squared training set 0.999236791700946


In [ ]:
#df_correct_preds.filter((pl.col("Anomaly") == 2))

Ale importance transakce 8891:  
[ 62  56  86  85 120  45  94  31  32 105]  
day_of_month_cos: 0.4909162578149944  
market_index_amount_diff_1: -0.2561160148612243  
rolling_min_7d: 0.19531048134345905  
rolling_min_24h: -0.17233210873780314  
InputOutput_freq_originator: 0.15725736811678412  
product_type_index_timestamp_diff_1: 0.12452001855439251  
rolling_mean_originator_7d: -0.09423044080411237  
day_of_month: 0.057788530885468  
hour: -0.05252906617590189  
rolling_min_originator_24h: 0.0374342169108247  

## 2.3. Security bought or sold at an unusual time:
It is unusual for clients to trade specific securities outside of a specific timeframe (for example, outside of the opening and closing times of a stock exchange).  


In [99]:
anomaly_type = 3
idx = 1260		
X_train_a = X_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(X_train_a.shape)
print(y_train_a.shape)

(11425, 121)
(11425,)


In [100]:
x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

(1, 121)
[[0.29441406 0.         1.         0.         1.         0.        ]]
3
3


In [101]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1260,3,27178334,2019-03-18 02:12:51,"""Client_066""",3,1,0.294414,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,1,18,2,12,false,2.693393,1.938064,1.604263,1.429005,1.32599,1.62011,-0.284955,-0.112912,0.0,0.0,1.036372,-0.917766,-0.698787,-0.576665,-0.484611,2.410622,-0.183119,-0.071133,0.0,0.0,-0.809817,-0.947825,-0.737582,2.488078,0.71708,1.365456,0.513469,-1.287407,-0.415434,3.620574,-1.291928,0.574992,-7.040985,-4.478681,-5.307024,-0.885715,-7.668171,-6.996847,-9.240966

In [102]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(X_train_n)
distances_n, indices_n = knn_normal.kneighbors(x, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(X_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(x, n_neighbors=k)
print(indices_a)
print(distances_a)

[[7043866 7043354 7043914 7043910 7043558 7043880 7043722 7043337 7043619]]
[[ 9.80910193  9.83282517 10.16924695 10.86467745 10.88809653 10.95037122
  11.10880725 11.11182644 11.14470368]]
[[3645 3690 3679 3686 3681 9436 3696 9439 3692]]
[[8.4496887  8.77086534 8.88993799 8.97371076 9.03059821 9.2971054
  9.364501   9.41529693 9.56759954]]


In [103]:
lasso_x_train = np.vstack((X_train_n[indices_n.flatten().tolist()], X_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))
print(lasso_x_train.shape)
print(lasso_y_train.shape)
lasso_classifier = Lasso(alpha=0.01)
lasso_classifier.fit(lasso_x_train, lasso_y_train)

(18, 121)
(18,)


Lasso(alpha=0.01)

In [108]:
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[116  65  62  46  98  51  93  36 120 112]
InputOutput_freq_originator: 0.7002027325078276
rolling_mean_24h: 0.44943992992865894
minute_cos: -0.34185366174871246
product_class_index_timestamp_diff_1: 0.19978465253480407
rolling_max_originator_7d: 0.06519587949834159
originator_index_amount_diff_1: -0.04534185307085977
rolling_std_originator_24h: 0.04198310245251187
market_index_timestamp_diff_1: 0.017506986342667206
Currency_freq_originator: -0.0
cumcount_Market: 0.0


In [109]:
imp_features = [feature_names[idx] for idx in sorted_importance[:10]]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['InputOutput_freq_originator', 'rolling_mean_24h', 'minute_cos', 'product_class_index_timestamp_diff_1', 'rolling_max_originator_7d', 'originator_index_amount_diff_1', 'rolling_std_originator_24h', 'market_index_timestamp_diff_1', 'Currency_freq_originator', 'cumcount_Market']


InputOutput_freq_originator,rolling_mean_24h,minute_cos,product_class_index_timestamp_diff_1,rolling_max_originator_7d,originator_index_amount_diff_1,rolling_std_originator_24h,market_index_timestamp_diff_1,Currency_freq_originator,cumcount_Market
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
5.86853,-5.307024,0.574992,2.410622,0.846607,-0.809817,-13.098606,1.62011,0.617451,0.098906


In [105]:
# Pro alpha = 0.05
lasso_classifier = Lasso(alpha=0.05)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[116  65  62  46  61  51   4  93 120  98]
InputOutput_freq_originator: 0.7013087267132068
rolling_mean_24h: 0.4454291968333194
minute_cos: -0.34699151384122495
product_class_index_timestamp_diff_1: 0.17918565999176703
minute_sin: -0.14234736344503757
originator_index_amount_diff_1: -0.12232507012927467
Market_2: 0.11441864940632246
rolling_std_originator_24h: 0.06323571192360138
Currency_freq_originator: -0.061281951625473854
rolling_max_originator_7d: 0.02488112078625296


In [106]:
# Pro alpha = 0.1
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[116  65  62  46  98  51  93  36 120 112]
InputOutput_freq_originator: 0.7002027325078276
rolling_mean_24h: 0.44943992992865894
minute_cos: -0.34185366174871246
product_class_index_timestamp_diff_1: 0.19978465253480407
rolling_max_originator_7d: 0.06519587949834159
originator_index_amount_diff_1: -0.04534185307085977
rolling_std_originator_24h: 0.04198310245251187
market_index_timestamp_diff_1: 0.017506986342667206
Currency_freq_originator: -0.0
cumcount_Market: 0.0


## 2.4. Large asset withdrawal:
A sudden spike in transaction amount withdrawn from an account or transferred out, which deviates from the previous transactional activity and is absent of any commercial rationale or related corporate action event.  

In [149]:
anomaly_type = 4
idx = 16483
X_train_a = X_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(X_train_a.shape)
print(y_train_a.shape)

(60973, 121)
(60973,)


In [150]:
x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

(1, 121)
[[0.46960018 0.         1.         0.         0.         0.        ]]
4
4


In [151]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
16483,4,27012263,2019-03-18 07:21:58,"""Client_066""",4,1,0.4696,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,1,18,7,21,false,2.694239,1.938699,1.604809,1.429475,1.326425,0.874352,3.617136,9.14322,0.0,0.0,1.037095,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-1.023369,1.803247,1.229654,0.000056,0.71708,1.365456,0.513469,-1.287407,-1.409974,1.270989,-1.224723,-0.707304,-0.280875,-1.4086,-1.684538,-0.895477,-1.64794,-3.436343,-5.080934,0.20

In [152]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(X_train_n)
distances_n, indices_n = knn_normal.kneighbors(x, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(X_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(x, n_neighbors=k)
print(indices_a)
print(distances_a)

[[14138205 14135088 11855308 14135887 14147730 11836983 14144681 14148504
  11838074]]
[[8.88116915 8.90100421 9.28912904 9.31328498 9.45472792 9.5022187
  9.54287454 9.56637248 9.57343492]]
[[45052 45443  5854 36790 39608 26430 45723 41654 26385]]
[[ 9.6071285  10.40947282 11.20253104 11.28701603 11.35696534 11.43945213
  11.51885464 11.58405372 11.58416456]]


In [153]:
lasso_x_train = np.vstack((X_train_n[indices_n.flatten().tolist()], X_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))
print(lasso_x_train.shape)
print(lasso_y_train.shape)
lasso_classifier = Lasso(alpha=0.01)
lasso_classifier.fit(lasso_x_train, lasso_y_train)

(18, 121)
(18,)


Lasso(alpha=0.01)

In [154]:
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[  1  65  68 119 116  58  69   2 115 111]
InputOutput_Buy: -3.8145697144400788
rolling_mean_24h: 0.028400094582235892
rolling_sum_12h: 0.01957081807722386
Product Class_freq_originator: -0.011515917183301576
InputOutput_freq_originator: 0.010195188613114371
day_of_month_cos: -0.0018328419977957386
rolling_sum_24h: 0.0010529616193847367
InputOutput_Sell: 1.1842378929335002e-15
cumcount_Currency: -0.0
cumcount_InputOutput: -0.0


In [155]:
y_pred = lasso_classifier.predict(x)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[3.89945152]
R squared training set 0.9997168872522549


In [156]:
imp_features = [feature_names[idx] for idx in sorted_importance[:10]]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['InputOutput_Buy', 'rolling_mean_24h', 'rolling_sum_12h', 'Product Class_freq_originator', 'InputOutput_freq_originator', 'day_of_month_cos', 'rolling_sum_24h', 'InputOutput_Sell', 'cumcount_Currency', 'cumcount_InputOutput']


InputOutput_Buy,rolling_mean_24h,rolling_sum_12h,Product Class_freq_originator,InputOutput_freq_originator,day_of_month_cos,rolling_sum_24h,InputOutput_Sell,cumcount_Currency,cumcount_InputOutput
u8,f64,f64,f64,f64,f64,f64,u8,f64,f64
0,-1.684538,-3.436343,0.252304,5.883999,-1.287407,-5.080934,1,-0.300857,0.29178


## 2.5. An unusually large amount of collateral transferred in and out of an account within a short period of time:
This behavior is unusual as a client would not be able to invest by simply trading collateral, or at least such a strategy would be unusual.  


In [175]:
idx = 37661 #2762376
anomaly_type = 5
X_train_a = X_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(X_train_a.shape)
print(y_train_a.shape)

(68173, 121)
(68173,)


In [176]:
x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

(1, 121)
[[-0.05676139  1.          0.          1.          0.          0.        ]]
5
5


In [161]:
df_correct_preds.filter(pl.col("index") == 2762376)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2762376,5,29315121,2019-03-25 20:57:30,"""Client_066""",5,1,0.313846,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,25,20,57,false,2.722976,1.960468,1.622718,1.445076,1.340454,-0.669069,-0.284955,-0.112912,0.0,0.0,1.04478,-0.917766,-0.698787,-0.576665,-0.484611,2.420729,-0.183119,-0.071133,0.0,0.0,0.742409,-0.000208,-1.474181,-2.09058,0.71708,1.365456,1.493561,0.276227,1.227968,1.900558,0.575268,1.291715,-0.404574,-0.763154,-0.902047,-0.807294,-0.01851,1.159393,0.545971,

In [162]:
df_correct_preds.filter(pl.col("index") == 37661)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
37661,5,27267059,2019-03-18 08:00:06,"""Client_269""",5,1,-0.056761,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,18,8,0,false,2.706488,1.947968,1.612474,1.436145,1.332426,-0.669069,-0.284955,-0.112912,0.0,0.0,1.097752,1.392918,1.733295,2.015149,-0.484611,2.410997,-0.183119,-0.071133,0.0,0.0,-0.984395,-0.000208,1.113591,-2.277635,0.71708,1.365456,0.513469,-1.287407,-1.364252,0.64142,0.147873,1.40623,0.540915,0.35138,0.392118,-0.925693,0.136732,-2.270245,-3.718484,0.204,

In [177]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(X_train_n)
distances_n, indices_n = knn_normal.kneighbors(x, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(X_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(x, n_neighbors=k)
print(indices_a)
print(distances_a)

[[21808797 16848393 11909298 14165864 11900477 16845152 11902233 14172080
  19210457]]
[[5.97221627 6.12604803 6.21125437 6.22671549 6.33028924 6.42095239
  6.42564523 6.43306315 6.51636174]]
[[64257 56575 56851 57051 64249 56429 50186 57147 50328]]
[[6.26033484 6.34928418 6.44157577 6.44399835 6.46945057 6.53568155
  6.61234576 6.62052969 6.63836741]]


In [178]:
lasso_x_train = np.vstack((X_train_n[indices_n.flatten().tolist()], X_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))
print(lasso_x_train.shape)
print(lasso_y_train.shape)
lasso_classifier = Lasso(alpha=0.01)
lasso_classifier.fit(lasso_x_train, lasso_y_train)

(18, 121)
(18,)


Lasso(alpha=0.01)

In [173]:
# id = 2762376
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[ 46  54  78  61 119 115 118 117 116 111]
product_class_index_timestamp_diff_1: 1.728415672604561
product_class_index_amount_diff_1: -0.08604077684077303
rolling_max_7d: -0.018608134257282374
minute_sin: -0.0014934346638327844
Product Class_freq_originator: 0.0
cumcount_Currency: -0.0
Product Type_freq_originator: -0.0
Market_freq_originator: 0.0
InputOutput_freq_originator: -0.0
cumcount_InputOutput: -0.0


In [166]:
# id = 37661
sorted_importance = show_importance(lasso_classifier.coef_, feature_names, n_features=10)

[ 18  73  25   8 109 116  46  31  44  51]
Product Type_FutureEquity: 2.7341464131427853
rolling_std_24h: 2.403818357595594
Product Class_Cash in / out (withdrawal), Security in / out: 2.0495135948153917
Product Type_Bond: -1.759848693811727
rolling_n_transactions_originator_24h: -1.5875966002566642
InputOutput_freq_originator: -1.3203312943447312
product_class_index_timestamp_diff_1: -0.7366671992633759
originator_index_timestamp_diff_1: -0.47146439591708034
product_type_index_timestamp_diff_4: 0.44859961751598837
originator_index_amount_diff_1: -0.34958264889641805


In [167]:
y_pred = lasso_classifier.predict(x)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[1.45425859]
R squared training set 0.9986235820448766


In [174]:
# index = 2762376
imp_features = [feature_names[idx] for idx in sorted_importance[:10]]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['product_class_index_timestamp_diff_1', 'product_class_index_amount_diff_1', 'rolling_max_7d', 'minute_sin', 'Product Class_freq_originator', 'cumcount_Currency', 'Product Type_freq_originator', 'Market_freq_originator', 'InputOutput_freq_originator', 'cumcount_InputOutput']


product_class_index_timestamp_diff_1,product_class_index_amount_diff_1,rolling_max_7d,minute_sin,Product Class_freq_originator,cumcount_Currency,Product Type_freq_originator,Market_freq_originator,InputOutput_freq_originator,cumcount_InputOutput
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2.420729,-2.09058,-0.728241,0.575268,0.248525,0.352475,0.263391,-0.320222,-6.5257,-0.022088


In [168]:
# idx = 37661
imp_features = [feature_names[idx] for idx in sorted_importance[:10]]
print(imp_features)
df_test.filter(pl.col('index') == idx).select(imp_features)

['Product Type_FutureEquity', 'rolling_std_24h', 'Product Class_Cash in / out (withdrawal), Security in / out', 'Product Type_Bond', 'rolling_n_transactions_originator_24h', 'InputOutput_freq_originator', 'product_class_index_timestamp_diff_1', 'originator_index_timestamp_diff_1', 'product_type_index_timestamp_diff_4', 'originator_index_amount_diff_1']


Product Type_FutureEquity,rolling_std_24h,"Product Class_Cash in / out (withdrawal), Security in / out",Product Type_Bond,rolling_n_transactions_originator_24h,InputOutput_freq_originator,product_class_index_timestamp_diff_1,originator_index_timestamp_diff_1,product_type_index_timestamp_diff_4,originator_index_amount_diff_1
u8,f64,u8,u8,f64,f64,f64,f64,f64,f64
0,0.267433,1,0,-2.395277,0.542062,2.410997,2.706488,2.015149,-0.984395


# 3. Explanations on input space

In [182]:
z_train_n = z_train[y_pred_train == 0]
y_train_n = y_pred_train[y_pred_train == 0]
print(z_train_n.shape)
print(y_train_n.shape)

(24291851, 16)
(24291851,)


## 3.1. Small but highly frequent transactions generated within a short timeframe:
A pattern that contains multiple transactions below applicable reporting thresholds.  

In [398]:
anomaly_type = 1
#idx = 8357	
#idx = 47208
#idx = 59510
#idx = 51542
k = 9

In [399]:
z_train_a = z_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(z_train_a.shape)
print(y_train_a.shape)

(16437, 16)
(16437,)


In [400]:
x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
#x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

model.encoder.eval()
test_transaction_latent = model.encoder(x)
test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
test_transaction_latent = test_transaction_latent.reshape(1, -1)
test_transaction_latent.shape

torch.Size([1, 121])
tensor([[-1.3714,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
1
1


(1, 16)

In [393]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
59510,1,27375815,2019-03-18 08:38:54,"""Client_269""",1,1,-1.40766,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,1,18,8,38,false,2.192313,1.948054,1.612504,1.43617,1.332452,1.661113,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-1.06959,-1.788361,-0.000167,0.000056,0.71708,1.365456,0.513469,-1.287407,-1.364252,0.64142,0.946355,-1.051156,0.724129,0.360025,0.402319,-0.895129,0.154832,-1.651899,-2.996019,0

In [401]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(z_train_n)
distances_n, indices_n = knn_normal.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(z_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_a)
print(distances_a)

[[ 8469041   786623 14386848 16456170  1653971 14270323  2600641  6884144
    637531]]
[[5.34924221 5.54619646 5.6588006  5.72003078 5.72298956 5.72617292
  5.77509403 5.79466438 5.8052392 ]]
[[ 5630  5573 13153 12428  5596  5594 13160  5592  4047]]
[[1.00450015 1.06799412 1.07400823 1.0847646  1.17012119 1.17083704
  1.22067535 1.24128497 1.25460839]]


In [402]:
lasso_x_train = np.vstack((z_train_n[indices_n.flatten().tolist()], z_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))

print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 16)
(18,)


In [403]:
lasso_classifier = Lasso(alpha=0.05)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[ 0.0273791   0.         -0.10534227  0.         -0.          0.
 -0.         -0.         -0.          0.          0.19681309 -0.
  0.         -0.         -0.10328226  0.        ]


In [195]:
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[ 4  7 13  8 10  0 11  1 15  9]
z_4: -2.842630624771118
z_7: -1.797913908958435
z_13: -1.7776914834976196
z_8: -0.8401194214820862
z_10: 0.7909547090530396
z_0: -0.6901487708091736
z_11: -0.5478876233100891
z_1: 0.5196952819824219
z_15: -0.3222387135028839
z_9: -0.0


In [381]:
#idx = 47208
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[13 14 10  4 12 11  9 15  8  7]
z_13: -0.399502694606781
z_14: -0.183762788772583
z_10: 0.12452422827482224
z_4: -0.0983685553073883
z_12: 0.0
z_11: -0.0
z_9: 0.0
z_15: -0.0
z_8: -0.0
z_7: -0.0


In [397]:
#idx = 59510
# 59510 a 47208 - stejny klient - vysvetleni jsou podobna
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[13 14 10 15 12 11  9  8  7  6]
z_13: -0.624029815196991
z_14: -0.10480958968400955
z_10: 0.057289741933345795
z_15: 0.0
z_12: -0.0
z_11: -0.0
z_9: 0.0
z_8: -0.0
z_7: -0.0
z_6: 0.0


In [404]:
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[10  2 14  0 12 11 13 15  8  9]
z_10: 0.19681309163570404
z_2: -0.10534226894378662
z_14: -0.10328225791454315
z_0: 0.027379101142287254
z_12: 0.0
z_11: -0.0
z_13: -0.0
z_15: 0.0
z_8: -0.0
z_9: 0.0


In [197]:
y_pred = lasso_classifier.predict(test_transaction_latent)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[3.9602852]
R squared training set 0.7354124784469604


## 3.2. Transactions with rounded normalized amounts bought or sold within an account:
It is unusual for transactions in capital markets to have rounded amounts (unless they occur in markets where foreign exchange conversion causes rounding errors).  


In [253]:
anomaly_type = 2
idx = 2762998
#idx = 8891
#idx = 9186

In [254]:
z_train_a = z_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(z_train_a.shape)
print(y_train_a.shape)

(3075, 16)
(3075,)


In [255]:
x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
#x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

model.encoder.eval()
test_transaction_latent = model.encoder(x)
test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
test_transaction_latent = test_transaction_latent.reshape(1, -1)
test_transaction_latent.shape

torch.Size([1, 121])
tensor([[-0.1537,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000]])
2
2


(1, 16)

In [256]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2762998,2,29358356,2019-03-25 20:58:44,"""Client_126""",2,1,-0.153726,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,1,25,20,58,true,1.976088,1.405621,1.183206,1.062452,0.996917,1.666501,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,-0.92282,-1.812988,-0.000167,0.000056,0.71708,1.365456,1.493561,0.276227,1.227968,1.900558,0.437069,1.344762,-0.20075,-0.763227,-0.89009,-0.796219,-0.016536,1.158688,0.548016

In [257]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(z_train_n)
distances_n, indices_n = knn_normal.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(z_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_a)
print(distances_a)

[[ 7024447 16785908 15389272 17173201 18709443  1985380 13342260 10973830
  22879734]]
[[2.38548064 2.74158287 2.77904725 2.78241467 2.7916584  2.81790018
  2.83059764 2.8639226  2.88578558]]
[[2003 1680 2585 2584 2852 2334 2028 2609 2331]]
[[1.31661534 1.47595024 1.56430936 1.58656859 1.62715888 1.63853288
  1.68013847 1.69776201 1.70690382]]


In [258]:
lasso_x_train = np.vstack((z_train_n[indices_n.flatten().tolist()], z_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))

print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 16)
(18,)


In [214]:
# idx = 8891
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[-0.          0.         -0.         -0.         -0.11740979  0.
  0.         -0.         -0.          0.14261469  0.         -0.
 -0.          0.         -0.59548354 -0.        ]


In [251]:
#idx = 9186
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[-0.          0.         -0.         -0.         -0.04059238  0.
  0.         -0.         -0.          0.07508865  0.         -0.
 -0.          0.06717732 -0.67675436 -0.        ]


In [259]:
#idx = 2762998
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[-0.          0.          0.          0.4288372  -0.21016456 -0.
  0.         -0.         -0.          0.          0.07228045 -0.
 -0.          0.         -0.7131233   0.        ]


In [218]:
# idx = 8891
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[14  9  4 13 12 11 10 15  8  7]
z_14: -0.5954835414886475
z_9: 0.1426146924495697
z_4: -0.11740978807210922
z_13: 0.0
z_12: -0.0
z_11: -0.0
z_10: 0.0
z_15: -0.0
z_8: -0.0
z_7: -0.0


In [252]:
#idx = 9186
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[14  9 13  4 12 11 10 15  8  7]
z_14: -0.676754355430603
z_9: 0.07508864998817444
z_13: 0.06717731803655624
z_4: -0.04059237986803055
z_12: -0.0
z_11: -0.0
z_10: 0.0
z_15: -0.0
z_8: -0.0
z_7: -0.0


In [261]:
#idx = 2762998
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[14  3  4 10 12 11 13 15  8  9]
z_14: -0.7131233215332031
z_3: 0.42883720993995667
z_4: -0.21016456186771393
z_10: 0.07228045165538788
z_12: -0.0
z_11: -0.0
z_13: 0.0
z_15: 0.0
z_8: -0.0
z_9: 0.0


In [217]:
y_pred = lasso_classifier.predict(test_transaction_latent)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[1.8182106]
R squared training set 0.9356386065483093


## 3.3. Security bought or sold at an unusual time:
It is unusual for clients to trade specific securities outside of a specific timeframe (for example, outside of the opening and closing times of a stock exchange).  


In [295]:
anomaly_type = 3
#idx = 1260		
#idx = 1157
idx = 2191296
z_train_a = z_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(z_train_a.shape)
print(y_train_a.shape)

(11425, 16)
(11425,)


In [296]:
x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
#x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

model.encoder.eval()
test_transaction_latent = model.encoder(x)
test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
test_transaction_latent = test_transaction_latent.reshape(1, -1)
test_transaction_latent.shape

torch.Size([1, 121])
tensor([[0.4377, 0.0000, 1.0000, 1.0000, 0.0000, 0.0000]])
3
3


(1, 16)

In [297]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2191296,3,29582320,2019-03-25 04:55:24,"""Client_066""",3,1,0.437687,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,1,25,4,55,false,2.720786,1.958762,1.621336,1.443876,1.339378,-0.669069,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,1.014191,-0.000208,-0.000167,0.000056,0.71708,1.365456,1.493561,0.276227,-1.016961,2.991005,0.831314,1.143892,4.9937,7.037661,8.281472,-0.384886,-6.23429,-5.238993,-7.187119,

In [298]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(z_train_n)
distances_n, indices_n = knn_normal.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(z_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_a)
print(distances_a)

[[19181254  4550302 19181207  4550311  4550573 19181273  4551119  4549768
  19181228]]
[[2.00121045 2.05460787 2.06765532 2.07407093 2.07464314 2.09097219
  2.09130216 2.11779213 2.12236404]]
[[ 5888  4770  5929  5902  5856  5883  5868 10495  4838]]
[[1.13939023 1.14369786 1.3032707  1.35032642 1.36107588 1.36896372
  1.40083265 1.40411448 1.41559124]]


In [299]:
lasso_x_train = np.vstack((z_train_n[indices_n.flatten().tolist()], z_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))

print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 16)
(18,)


In [300]:
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[ 0.         -0.          0.         -0.         -0.          0.
 -0.00651955 -0.          0.          0.          2.3100157   0.
 -0.          0.         -0.6542186   0.        ]


In [270]:
# idx = 1260
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[15  9 14 13 12 11 10  8  7  6]
z_15: 1.6302036046981812
z_9: 1.4055988788604736
z_14: -0.5636571645736694
z_13: 0.0
z_12: 0.0
z_11: 0.0
z_10: 0.0
z_8: -0.0
z_7: -0.0
z_6: -0.0


In [278]:
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[15  4  3  8 12 11 13 14  9 10]
z_15: 1.805695652961731
z_4: -1.358310580253601
z_3: 1.0342575311660767
z_8: 0.10734690725803375
z_12: -0.0
z_11: 0.0
z_13: 0.0
z_14: -0.0
z_9: 0.0
z_10: 0.0


In [301]:
#idx= 2191296
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[10 14  6 13 12 11  9 15  8  7]
z_10: 2.3100156784057617
z_14: -0.6542186141014099
z_6: -0.006519550457596779
z_13: 0.0
z_12: -0.0
z_11: 0.0
z_9: 0.0
z_15: 0.0
z_8: 0.0
z_7: -0.0


## 2.4. Large asset withdrawal:
A sudden spike in transaction amount withdrawn from an account or transferred out, which deviates from the previous transactional activity and is absent of any commercial rationale or related corporate action event.  


In [333]:
anomaly_type = 4
#idx = 16483
#idx = 43121
idx = 1578200
z_train_a = z_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(z_train_a.shape)
print(y_train_a.shape)

(60973, 16)
(60973,)


In [334]:
x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
#x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

model.encoder.eval()
test_transaction_latent = model.encoder(x)
test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
test_transaction_latent = test_transaction_latent.reshape(1, -1)
test_transaction_latent.shape

torch.Size([1, 121])
tensor([[0.2837, 0.0000, 1.0000, 0.0000, 1.0000, 0.0000]])
4
4


(1, 16)

In [335]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1578200,4,28389041,2019-03-21 13:52:58,"""Client_126""",4,1,0.283702,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,4,21,13,52,false,1.868977,1.324096,1.099986,0.991582,0.93298,-0.669069,-0.284955,-0.112912,0.0,0.0,-1.390533,-0.917766,-0.698787,-0.576665,-0.484611,-0.433523,-0.183119,-0.071133,0.0,0.0,0.456582,-0.000208,-0.000167,0.000056,-0.1313,-1.087529,1.212389,-0.80366,-0.068142,-1.16148,1.144188,0.831028,-0.071814,-0.968032,-0.420084,-0.36045,0.033574,0.121585,0.3093

In [336]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(z_train_n)
distances_n, indices_n = knn_normal.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(z_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_a)
print(distances_a)

[[22637796  6515247 14918450 21045242 23973410 18005843 18596616 17494687
  24128359]]
[[2.09333301 2.20503902 2.21301699 2.23646998 2.27940512 2.28431606
  2.3295927  2.33245277 2.35924649]]
[[51779 51782 51854 51914 51770 51685 51789 51686 56578]]
[[0.6487757  0.66346639 0.7476024  0.81965625 0.82133055 0.83718187
  0.86397141 0.86418015 0.87199795]]


In [337]:
lasso_x_train = np.vstack((z_train_n[indices_n.flatten().tolist()], z_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))

print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 16)
(18,)


In [338]:
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[-0.         -0.         -0.          0.68520284 -1.7985528   0.
 -0.         -0.         -0.          0.          0.         -0.51679516
  0.         -0.9433232  -0.         -0.39799213]


In [313]:
# idx = 16483
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[14  2 11 15 12 10  9 13  8  7]
z_14: -0.8668021559715271
z_2: -0.6827273368835449
z_11: -0.5504565238952637
z_15: -0.12361042946577072
z_12: 0.0
z_10: 0.0
z_9: 0.0
z_13: -0.0
z_8: 0.0
z_7: -0.0


In [330]:
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[11 15 13  3 14  0 12 10  8  9]
z_11: -1.0144938230514526
z_15: -0.3820912539958954
z_13: -0.20467519760131836
z_3: 0.18361549079418182
z_14: -0.13533329963684082
z_0: -0.08610198646783829
z_12: 0.0
z_10: 0.0
z_8: 0.0
z_9: -0.0


In [339]:
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[ 4 13  3 11 15 10 12 14  8  9]
z_4: -1.7985527515411377
z_13: -0.9433231949806213
z_3: 0.6852028369903564
z_11: -0.5167951583862305
z_15: -0.3979921340942383
z_10: 0.0
z_12: 0.0
z_14: -0.0
z_8: -0.0
z_9: 0.0


In [340]:
y_pred = lasso_classifier.predict(test_transaction_latent)
print(y_pred)
print('R squared training set', lasso_classifier.score(lasso_x_train, lasso_y_train))

[3.3733356]
R squared training set 0.9586156010627747


## 3.5. An unusually large amount of collateral transferred in and out of an account within a short period of time:
This behavior is unusual as a client would not be able to invest by simply trading collateral, or at least such a strategy would be unusual.   

In [361]:
# idx = 37661
# idx = 2762376
idx = 2742220
anomaly_type = 5
z_train_a = z_train[y_pred_train == anomaly_type]
y_train_a = y_pred_train[y_pred_train == anomaly_type]
print(z_train_a.shape)
print(y_train_a.shape)

(68173, 16)
(68173,)


In [362]:
x = torch.tensor(X_test[idx], dtype=torch.float32).reshape(1, -1)
#x = X_test[idx].reshape(1, -1)

print(x.shape)
print(x[:, :6])
print(y_test[idx])
print(y_pred_test[idx])

model.encoder.eval()
test_transaction_latent = model.encoder(x)
test_transaction_latent = test_transaction_latent.cpu().detach().numpy()
test_transaction_latent = test_transaction_latent.reshape(1, -1)
test_transaction_latent.shape

torch.Size([1, 121])
tensor([[0.1276, 1.0000, 0.0000, 1.0000, 0.0000, 0.0000]])
5
5


(1, 16)

In [363]:
df_correct_preds.filter(pl.col("index") == idx)

index,Model Prediction,id,datetime,Originator,Anomaly,Anomaly_bin,amount_log,InputOutput_Buy,InputOutput_Sell,Market_1,Market_2,Market_3,Market_4,Product Type_ADR,Product Type_Bond,Product Type_CAADR,Product Type_ETOEquity,Product Type_ETOEquityIndex,Product Type_Equity,Product Type_FX,Product Type_FXForward,Product Type_FXSwap,Product Type_FutureBond,Product Type_FutureCommodity,Product Type_FutureEquity,Product Type_FutureEquityIndex,Product Type_FutureFX,Product Type_FutureOptionEquityIndex,Product Type_Repo,Product Type_SimpleTransfer,Product Class_ADR Conversion,"Product Class_Cash in / out (withdrawal), Security in / out",Product Class_External fee,Product Class_Trade,Currency_1,Currency_2,day_of_week,day_of_month,hour,minute,is_rounded,originator_index_timestamp_diff_1,originator_index_timestamp_diff_2,originator_index_timestamp_diff_3,originator_index_timestamp_diff_4,originator_index_timestamp_diff_5,market_index_timestamp_diff_1,market_index_timestamp_diff_2,market_index_timestamp_diff_3,market_index_timestamp_diff_4,market_index_timestamp_diff_5,product_type_index_timestamp_diff_1,product_type_index_timestamp_diff_2,product_type_index_timestamp_diff_3,product_type_index_timestamp_diff_4,product_type_index_timestamp_diff_5,product_class_index_timestamp_diff_1,product_class_index_timestamp_diff_2,product_class_index_timestamp_diff_3,product_class_index_timestamp_diff_4,product_class_index_timestamp_diff_5,originator_index_amount_diff_1,market_index_amount_diff_1,product_type_index_amount_diff_1,product_class_index_amount_diff_1,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,hour_sin,hour_cos,minute_sin,minute_cos,rolling_mean_1h,rolling_mean_12h,rolling_mean_24h,rolling_mean_7d,rolling_sum_1h,rolling_sum_12h,rolling_sum_24h,rolling_sum_7d,rolling_std_1h,rolling_std_12h,rolling_std_24h,rolling_std_7d,rolling_max_1h,rolling_max_12h,rolling_max_24h,rolling_max_7d,rolling_min_1h,rolling_min_12h,rolling_min_24h,rolling_min_7d,rolling_n_transactions_1h,rolling_n_transactions_12h,rolling_n_transactions_24h,rolling_n_transactions_7d,rolling_mean_originator_1h,rolling_mean_originator_12h,rolling_mean_originator_24h,rolling_mean_originator_7d,rolling_std_originator_1h,rolling_std_originator_12h,rolling_std_originator_24h,rolling_std_originator_7d,rolling_max_originator_1h,rolling_max_originator_12h,rolling_max_originator_24h,rolling_max_originator_7d,rolling_min_originator_1h,rolling_min_originator_12h,rolling_min_originator_24h,rolling_min_originator_7d,rolling_sum_originator_1h,rolling_sum_originator_12h,rolling_sum_originator_24h,rolling_sum_originator_7d,rolling_n_transactions_originator_1h,rolling_n_transactions_originator_12h,rolling_n_transactions_originator_24h,rolling_n_transactions_originator_7d,cumcount_InputOutput,cumcount_Market,cumcount_Product Type,cumcount_Product Class,cumcount_Currency,InputOutput_freq_originator,Market_freq_originator,Product Type_freq_originator,Product Class_freq_originator,Currency_freq_originator
u32,i64,u32,datetime[μs],str,i64,i8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2742220,5,29315163,2019-03-25 20:19:22,"""Client_066""",5,1,0.127558,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,25,20,19,false,2.722951,1.96039,1.622662,1.44503,1.340415,-0.669069,-0.284955,-0.112912,0.0,0.0,1.041646,1.371174,-0.698787,-0.576665,-0.484611,2.420702,-0.183119,-0.071133,0.0,0.0,-0.984276,-0.000208,-1.487097,-2.44517,0.71708,1.365456,1.493561,0.276227,1.227968,1.900558,-1.344977,-0.437218,-1.523585,-0.726696,-0.866729,-0.731116,0.209802,1.177584,0.490627

In [364]:
knn_normal = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_normal.fit(z_train_n)
distances_n, indices_n = knn_normal.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_n)
print(distances_n)

knn_anomaly = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
knn_anomaly.fit(z_train_a)
distances_a, indices_a = knn_anomaly.kneighbors(test_transaction_latent, n_neighbors=k)
print(indices_a)
print(distances_a)

[[2224812 2445974 2486129 2863133 2452079 3157412 2840016 1799853  537756]]
[[3.30597401 3.36121106 3.45149326 3.50405145 3.50886726 3.53707814
  3.54500651 3.54833174 3.55391145]]
[[36947 64918 64916 64913 32603 29873 19184 32601 64888]]
[[0.73515326 0.8501938  0.88848692 0.93049419 0.94628245 0.96698058
  0.97500175 0.97839004 1.01206148]]


In [365]:
lasso_x_train = np.vstack((z_train_n[indices_n.flatten().tolist()], z_train_a[indices_a.flatten().tolist()]))
lasso_y_train = np.concatenate((y_train_n[indices_n.flatten().tolist()], y_train_a[indices_a.flatten().tolist()]))

print(lasso_x_train.shape)
print(lasso_y_train.shape)

(18, 16)
(18,)


In [366]:
lasso_classifier = Lasso(alpha=0.1)
lasso_classifier.fit(lasso_x_train, lasso_y_train)
print(lasso_classifier.coef_)
#print(lasso_classifier.intercept_)

[ 0.          0.81061304 -0.          0.         -0.          0.
 -0.1136227  -0.         -0.9288145   0.          0.         -0.83087265
  1.3570399  -0.         -0.38097972 -0.        ]


In [348]:
# idx = 16483
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[ 4  7 13  8 10  0 11  1 15  9]
z_4: -2.842630624771118
z_7: -1.797913908958435
z_13: -1.7776914834976196
z_8: -0.8401194214820862
z_10: 0.7909547090530396
z_0: -0.6901487708091736
z_11: -0.5478876233100891
z_1: 0.5196952819824219
z_15: -0.3222387135028839
z_9: -0.0


In [356]:
#idx = 2762376
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[ 6  4  8  1 12 14 13 15  9 10]
z_6: -1.2893599271774292
z_4: -1.1086502075195312
z_8: -1.0477161407470703
z_1: 0.3213449716567993
z_12: 0.31699803471565247
z_14: -0.08331853151321411
z_13: -0.0
z_15: -0.0
z_9: 0.0
z_10: 0.0


In [367]:
#idx = 2742220
sorted_importance = show_importance_latent(lasso_classifier.coef_, n_features=10)

[12  8 11  1 14  6 13 15  9 10]
z_12: 1.3570399284362793
z_8: -0.9288144707679749
z_11: -0.830872654914856
z_1: 0.8106130361557007
z_14: -0.3809797167778015
z_6: -0.11362269520759583
z_13: -0.0
z_15: -0.0
z_9: 0.0
z_10: 0.0
